In [1]:
from hamilton import driver
from hamilton.execution import executors

import rayfish
import rayfish.quantum_fisher_information as qfi
from rayfish.parameter_catalogue import common_transforms, distance_2_sources
from rayfish.quantum_fisher_information import incoherent_sources

dr = (
    driver.Builder()
    .enable_dynamic_execution(allow_experimental_mode=True)
    .with_modules(
        rayfish, qfi, incoherent_sources, common_transforms, distance_2_sources
    )
    .with_config({"dimension": 1})
    .with_local_executor(executors.SynchronousLocalTaskExecutor())
    .build()
)

In [2]:
import pandas as pd

In [3]:
inputs_ = {
    "wave_vector": 2.0,
    "rayleigh_length": 1.0,
    "number_of_point_source_distributions": 20,
    "maximal_length": 4.0,
    "minimal_spacing": 0.1,
}
result = dr.execute(
    final_vars=["coordinates", "spacing", "number_of_point_sources"], inputs=inputs_
)
distribution_df = pd.concat(
    [
        pd.DataFrame(result["spacing"], columns=["spacing"]),
        pd.DataFrame(
            result["coordinates"]  # , columns=["x1", "x2", "y1", "y2", "z1", "z2"]
        ),
    ],
    axis=1,
).set_index("spacing")

In [27]:
distribution_df

,0,1,2,3,4,5
spacing,,,,,,
0.1,-0.1,0.1,0.0,0.0,-0.1,0.1
0.2,-0.2,0.2,0.0,0.0,-0.2,0.2
0.3,-0.3,0.3,0.0,0.0,-0.3,0.3
0.4,-0.4,0.4,0.0,0.0,-0.4,0.4
0.5,-0.5,0.5,0.0,0.0,-0.5,0.5
0.6,-0.6,0.6,0.0,0.0,-0.6,0.6
0.7,-0.7,0.7,0.0,0.0,-0.7,0.7
0.8,-0.8,0.8,0.0,0.0,-0.8,0.8
0.9,-0.9,0.9,0.0,0.0,-0.9,0.9


In [ ]:
num_particles = result["number_of_point_sources"]
scene = {
    "xaxis": {"range": []},
    "yaxis": {"range": []},
    "zaxis": {"range": []},
}

for c_i, coord in enumerate(scene.keys()):
    min_coord = min([
        distribution_df.min(axis=0)[c_i * num_particles + i]
        for i in range(num_particles)
    ])
    max_coord = max([
        distribution_df.max(axis=0)[c_i * num_particles + i]
        for i in range(num_particles)
    ])
    scene[coord]["range"] = [min_coord - 0.5, max_coord + 0.5]

-2.0 2.0
0.0 0.0
-2.0 2.0


In [29]:
distribution_df.min(axis=0)

0   -2.0
1    0.1
2    0.0
3    0.0
4   -2.0
5    0.1
dtype: float64

In [26]:
df = distribution_df.loc[0.2]
df

0   -0.2
1    0.2
2    0.0
3    0.0
4   -0.2
5    0.2
Name: 0.2, dtype: float64

In [8]:
# Group columns by coordinate type (x, y, z)
num_particles = result["number_of_point_sources"]
particle_coordinates = {"x": [], "y": [], "z": []}

for c_i, coord in enumerate(particle_coordinates.keys()):
    particle_coordinates[coord] = [
        df[c_i * num_particles + i] for i in range(num_particles)
    ]

# Create the new DataFrame
new_df = pd.DataFrame(particle_coordinates)
new_df

,x,y,z
0,-0.2,0.0,-0.2
1,0.2,0.0,0.2


In [11]:
import plotly.express as px

fig = px.scatter_3d(
    new_df,
    x="x",
    y="y",
    z="z",
    color=new_df.index,  # Optional: differentiate points by a category
    title="3D Scatter Plot of Particles",
)
fig.show()